# API de precios de materiales y equipos eléctricos → Excel

**Archivo único** del proyecto `C:\APIS_PRECIOS`. Consulta **todos los materiales, equipos, categorías y precios**
de tiendas eléctricas colombianas y los guarda en **Excel**. Incluye una **API REST con Swagger**.

| | |
|---|---|
| **Tiendas** | 5 tiendas colombianas activas (tabla abajo) |
| **Fuente de datos** | Catálogos públicos de las tiendas: *Store API* de WooCommerce y `products.json` de Shopify (los mismos que usa cada tienda en su web) |
| **Salida** | Excel en `C:\APIS_PRECIOS\datos\` (hojas *Productos*, *Categorias*, *Resumen*) |
| **API / Swagger** | http://127.0.0.1:8001/docs |

## Cómo usarlo

1. Abrir este notebook en VS Code y elegir el kernel **.venv (Python 3.10)**.
2. Pulsar **Run All** → instala librerías, descarga todo el catálogo, guarda el Excel y levanta la API.
3. Abrir **http://127.0.0.1:8001/docs** → *Try it out* → *Execute* → **Download file**.

> El catálogo completo de todas las tiendas tarda varios minutos (miles de productos, 100 por petición,
> con pausa de 0,5 s entre peticiones para no cargar los servidores de las tiendas).

## Tiendas incluidas

| Clave | Tienda | Ciudad | Qué vende | Precios |
|---|---|---|---|---|
| `edelco` | Edelco | Ibagué / Bogotá | Materiales eléctricos hogar, industria, construcción, transformadores | Sí (IVA incluido) |
| `ineldec` | Ineldec | Cúcuta | Breakers, DPS, cables, transformadores, celdas MT, medidores, solar | Sí |
| `nalelectricos` | Nacional de Eléctricos | Bogotá | Cables y alambres, iluminación, control y motores, transformadores y accesorios de redes, breakers | Sí (IVA incluido) |
| `eimpsa` | EIMPSA Eléctricos Importados | Bogotá | Conductores de cobre y aluminio, tableros, encerramientos | Sí (IVA incluido) |
| `eys` | Distribuciones Eys | Bogotá | Ferretería con sección de eléctricos (cables, breakers, iluminación) | Sí (IVA incluido) |

**Configuradas pero desactivadas** (responden *403 – prohibido* a las consultas automáticas; quedan comentadas en `TIENDAS`):

| Clave | Tienda | Qué vende |
|---|---|---|
| `cys` | C&S Materiales Eléctricos (Shopify) | Tableros, automatización y control, cables, iluminación |
| `dyc` | D&C Distribuciones Eléctricas de Colombia, Bogotá | **Postes** de concreto, metálicos y fibra de vidrio, **herrajes**, aisladores, premoldeados (vende por cotización) |

> **Postes, herrajes y redes de distribución:** empresas como D&C Distribuciones, JD Eléctricos (Bogotá),
> Surtielectronic, Edemco (Medellín), Cables Colombia (Chía, conductores ACSR y MT) e Interelectricas los venden,
> pero no publican un catálogo abierto, venden por cotización o bloquean las consultas automáticas, por lo que no
> se pueden consultar desde aquí. Para estos productos la vía es pedirles la lista de precios (Excel) a cada empresa.

## Contenido

1. Instalación de librerías
2. Configuración
3. Funciones de consulta
4. Exportación a Excel
5. Ensayo: conexión, categorías, búsqueda y catálogo completo → Excel
6. API REST (FastAPI + Swagger)
7. Prueba automática de la API
8. Mantenimiento: agregar tiendas, errores frecuentes

## 1. Instalación de librerías

Solo instala lo que falte. Si es la primera vez, puede pedir **reiniciar el kernel** (botón *Restart*) y volver a pulsar *Run All*.

| Librería | Para qué |
|---|---|
| `requests` | Peticiones HTTP a las tiendas |
| `pandas`, `openpyxl` | Tablas y archivos Excel |
| `truststore` | Usa los certificados de Windows (evita el error SSL *self signed certificate* en redes con proxy/antivirus) |
| `fastapi`, `uvicorn` | API REST y Swagger |

In [ ]:
%pip install -q requests pandas openpyxl truststore fastapi uvicorn

## 2. Configuración

Aquí se ajusta todo lo editable:

- **`TIENDAS`**: tiendas a consultar. Cada una indica su plataforma (`woocommerce` o `shopify`) y si publica
  precios. Para agregar otra tienda basta una línea nueva (ver sección 8). Para dejar de consultar una tienda,
  comenta su línea con `#`.
- **`CARPETA_DATOS`**: dónde se guardan los Excel.
- **`PAUSA_SEG`**: espera entre peticiones (cortesía con los servidores).
- **`PUERTO_API`**: puerto de la API/Swagger.

In [ ]:
import html
import io
import json
import re
import threading
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

# Certificados de Windows: soluciona "SSLError: self signed certificate in certificate chain"
try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    print("Aviso: truststore no está instalado (ejecuta la celda de instalación).")

# Tiendas a consultar
#   tipo             -> plataforma de la tienda: "woocommerce" o "shopify"
#   precio_publicado -> False si la tienda vende por cotización (el precio queda vacío en el Excel)
TIENDAS = {
    "edelco":        {"nombre": "Edelco",                 "url": "https://edelco.co",                          "tipo": "woocommerce", "precio_publicado": True},
    "ineldec":       {"nombre": "Ineldec",                "url": "https://ineldec.com",                        "tipo": "woocommerce", "precio_publicado": True},
    "nalelectricos": {"nombre": "Nacional de Eléctricos", "url": "https://nalelectricos.com.co",               "tipo": "woocommerce", "precio_publicado": True},
    "eimpsa":        {"nombre": "EIMPSA",                 "url": "https://tienda.eimpsa.com",                  "tipo": "woocommerce", "precio_publicado": True},
    # Desactivadas: responden 403 (bloquean consultas automáticas). Se pueden probar quitando el '#'.
    # "cys":           {"nombre": "C&S Materiales Eléctricos", "url": "https://cysmaterialeselectricos.myshopify.com", "tipo": "shopify", "precio_publicado": True},
    "eys":           {"nombre": "Distribuciones Eys",     "url": "https://distribucioneseys.com",              "tipo": "woocommerce", "precio_publicado": True},
    # "dyc":           {"nombre": "D&C Distribuciones (postes y herrajes)", "url": "https://dycdistricolsas.com", "tipo": "woocommerce", "precio_publicado": False},
}

CARPETA_DATOS = Path(r"C:\APIS_PRECIOS\datos")   # salida de los Excel
CARPETA_DATOS.mkdir(parents=True, exist_ok=True)

PAUSA_SEG  = 0.5    # segundos entre peticiones
TIMEOUT    = 40     # segundos máximos por petición
POR_PAGINA = 100    # productos por petición (WooCommerce permite hasta 100; Shopify hasta 250)
PUERTO_API = 8001   # Swagger: http://127.0.0.1:8001/docs

# Columnas del Excel de productos (en este orden)
COLUMNAS = ["tienda", "id_producto", "nombre", "sku", "precio", "precio_regular", "precio_oferta",
            "en_oferta", "disponible", "categorias", "marca", "observacion", "url", "fecha_consulta"]

# Sesión HTTP con reintentos automáticos (cortes de conexión, 429/5xx): espera 2, 4, 8, 16 s
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=4, connect=4, read=4, backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"])))
session.headers.update({
    "User-Agent": "APIS_PRECIOS/1.0 (consulta de precios; uso interno)",
    "Accept": "application/json",
})
print("Configuración lista. Excel en:", CARPETA_DATOS)

## 3. Funciones de consulta

Cada tienda se consulta según su plataforma:

| Plataforma | Endpoint público | Devuelve |
|---|---|---|
| WooCommerce | `/wp-json/wc/store/v1/products` | Productos (nombre, SKU, precios, stock, categorías, enlace). Admite `search`, `category`, `page`, `per_page` |
| WooCommerce | `/wp-json/wc/store/v1/products/categories` | Categorías (id, nombre, padre, cantidad de productos) |
| Shopify | `/products.json?limit=250&page=N` | Productos con variantes (precio, SKU, disponibilidad), tipo de producto y marca |

- WooCommerce entrega precios en *unidades menores* (`currency_minor_unit`); se convierten a pesos.
- En Shopify la **categoría** es el *tipo de producto* (`product_type`) y la búsqueda se hace sobre el catálogo descargado.
- Si la tienda no publica precios (`precio_publicado=False`) o el precio es simbólico (≤ $1), el precio queda vacío
  y la columna `observacion` dice *"Precio no publicado (cotizar)"*.

| Función | Uso |
|---|---|
| `categorias_todas()` | Categorías de todas las tiendas (sección 5.2) |
| `probar_conexion(tienda)` | ¿Responde la tienda? ¿Cuántos productos tiene? |
| `categorias(tienda)` | Todas las categorías de una tienda |
| `productos(tienda, buscar=None, categoria=None, max_paginas=None)` | Productos con precio (todas las páginas si `max_paginas=None`) |
| `buscar_en_todas(texto)` | El mismo material en todas las tiendas, ordenado por precio |
| `catalogo_completo()` | **Todo**: productos y categorías de todas las tiendas |

In [ ]:
def _tienda(clave: str) -> dict:
    # Valida la clave de la tienda ('edelco', 'ineldec', ...).
    clave = clave.lower()
    if clave not in TIENDAS:
        raise KeyError(f"Tienda '{clave}' no configurada. Opciones: {', '.join(TIENDAS)}")
    return TIENDAS[clave]


def _get(url, params=None):
    # GET con control de errores y pausa de cortesía.
    r = session.get(url, params=params, timeout=TIMEOUT)
    r.raise_for_status()
    time.sleep(PAUSA_SEG)
    return r


def _texto(valor):
    # Quita etiquetas HTML y entidades (&amp; -> &).
    return html.unescape(re.sub(r"<[^>]+>", "", str(valor))).strip() if valor else valor


def _monto(valor, unidades_menores):
    # '125000' con currency_minor_unit=0 -> 125000.0
    if valor in (None, ""):
        return None
    try:
        return int(valor) / (10 ** int(unidades_menores or 0))
    except (TypeError, ValueError):
        return None


def _nota(clave, precio):
    # Devuelve (precio, observacion) aplicando la regla de precios no publicados.
    if not TIENDAS[clave].get("precio_publicado", True) or precio is None or precio <= 1:
        return None, "Precio no publicado (cotizar)"
    return precio, None


def _fila_woo(p, clave):
    # Convierte un producto de WooCommerce en una fila de la tabla.
    pr = p.get("prices") or {}
    m = pr.get("currency_minor_unit", 0)
    precio, obs = _nota(clave, _monto(pr.get("price"), m))
    return {
        "tienda": TIENDAS[clave]["nombre"],
        "id_producto": p.get("id"),
        "nombre": _texto(p.get("name")),
        "sku": p.get("sku") or None,
        "precio": precio,
        "precio_regular": _monto(pr.get("regular_price"), m) if precio else None,
        "precio_oferta": _monto(pr.get("sale_price"), m) if precio else None,
        "en_oferta": p.get("on_sale"),
        "disponible": p.get("is_in_stock"),
        "categorias": " | ".join(_texto(c.get("name")) for c in p.get("categories") or []),
        "marca": " | ".join(_texto(b.get("name")) for b in p.get("brands") or []) or None,
        "observacion": obs,
        "url": p.get("permalink"),
        "fecha_consulta": datetime.now().isoformat(timespec="seconds"),
    }


def _fila_shopify(p, clave):
    # Convierte un producto de Shopify (primera variante) en una fila de la tabla.
    v = (p.get("variants") or [{}])[0]
    def num(x):
        try:
            return float(x) if x not in (None, "") else None
        except ValueError:
            return None
    precio, obs = _nota(clave, num(v.get("price")))
    antes = num(v.get("compare_at_price"))
    en_oferta = bool(precio and antes and antes > precio)
    return {
        "tienda": TIENDAS[clave]["nombre"],
        "id_producto": p.get("id"),
        "nombre": _texto(p.get("title")),
        "sku": v.get("sku") or None,
        "precio": precio,
        "precio_regular": antes if en_oferta else precio,
        "precio_oferta": precio if en_oferta else None,
        "en_oferta": en_oferta,
        "disponible": any(x.get("available") for x in p.get("variants") or []),
        "categorias": p.get("product_type") or None,
        "marca": p.get("vendor") or None,
        "observacion": obs,
        "url": f"{TIENDAS[clave]['url']}/products/{p.get('handle')}",
        "fecha_consulta": datetime.now().isoformat(timespec="seconds"),
    }


def a_dataframe(filas) -> pd.DataFrame:
    # Lista de filas -> DataFrame con las columnas estándar.
    df = pd.DataFrame(filas)
    for c in COLUMNAS:
        if c not in df.columns:
            df[c] = None
    return df[COLUMNAS]


def probar_conexion(clave: str) -> dict:
    """Verifica que la tienda responda y cuántos productos publica."""
    t = _tienda(clave)
    try:
        if t["tipo"] == "shopify":
            r = session.get(f"{t['url']}/products.json", params={"limit": 1}, timeout=TIMEOUT)
            total = "sin dato (Shopify)"
        else:
            r = session.get(f"{t['url']}/wp-json/wc/store/v1/products", params={"per_page": 1}, timeout=TIMEOUT)
            total = r.headers.get("X-WP-Total")
        return {"tienda": t["nombre"], "ok": r.ok, "estado_http": r.status_code, "total_productos": total}
    except requests.RequestException as e:
        return {"tienda": t["nombre"], "ok": False, "error": type(e).__name__, "detalle": str(e)[:300]}


def _shopify_todos(clave, max_paginas=None, progreso=None):
    # Descarga todo el catálogo Shopify (250 productos por página).
    t = _tienda(clave)
    filas, pagina = [], 1
    while max_paginas is None or pagina <= max_paginas:
        datos = _get(f"{t['url']}/products.json", params={"limit": 250, "page": pagina}).json().get("products", [])
        filas += [_fila_shopify(p, clave) for p in datos]
        if progreso:
            progreso(t["nombre"], pagina, "?", len(filas))
        if len(datos) < 250:
            break
        pagina += 1
    return a_dataframe(filas)


def categorias(clave: str) -> pd.DataFrame:
    """Todas las categorías de una tienda (recorre todas las páginas)."""
    t = _tienda(clave)
    if t["tipo"] == "shopify":                      # en Shopify: tipos de producto
        df = _shopify_todos(clave)
        conteo = df["categorias"].fillna("(sin tipo)").value_counts()
        return pd.DataFrame({"tienda": t["nombre"], "id_categoria": None, "categoria": conteo.index,
                             "slug": conteo.index, "id_padre": None, "productos": conteo.values, "url": None})
    filas, pagina = [], 1
    while True:
        r = _get(f"{t['url']}/wp-json/wc/store/v1/products/categories",
                 params={"per_page": POR_PAGINA, "page": pagina})
        datos = r.json()
        filas += [{"tienda": t["nombre"], "id_categoria": c.get("id"), "categoria": _texto(c.get("name")),
                   "slug": c.get("slug"), "id_padre": c.get("parent"), "productos": c.get("count"),
                   "url": c.get("permalink")} for c in datos]
        if len(datos) < POR_PAGINA or pagina >= int(r.headers.get("X-WP-TotalPages", pagina)):
            break
        pagina += 1
    return pd.DataFrame(filas)


def productos(clave: str, buscar: str | None = None, categoria=None,
              max_paginas: int | None = None, progreso=None) -> pd.DataFrame:
    """
    Productos y precios de una tienda.
      buscar      -> texto (ej. 'cable 12'); deben aparecer todas las palabras
      categoria   -> id o slug de la categoría (ver categorias())
      max_paginas -> límite de páginas (None = todas)
      progreso    -> función opcional progreso(tienda, pagina, total_paginas, n_productos)
    """
    t = _tienda(clave)
    if t["tipo"] == "shopify":
        df = _shopify_todos(clave, None if (buscar or categoria) else max_paginas, progreso)
        if categoria not in (None, ""):
            df = df[df["categorias"].fillna("").str.lower() == str(categoria).lower()]
        if buscar:
            nombres = df["nombre"].fillna("").str.lower()
            for palabra in buscar.lower().split():
                df = df[nombres.loc[df.index].str.contains(palabra, regex=False)]
        return df.reset_index(drop=True)

    params = {"per_page": POR_PAGINA, "orderby": "title", "order": "asc"}
    if buscar:
        params["search"] = buscar
    if categoria not in (None, ""):
        params["category"] = categoria
    filas, pagina = [], 1
    while max_paginas is None or pagina <= max_paginas:
        r = _get(f"{t['url']}/wp-json/wc/store/v1/products", params={**params, "page": pagina})
        datos = r.json()
        total_paginas = int(r.headers.get("X-WP-TotalPages", 0) or 0)
        filas += [_fila_woo(p, clave) for p in datos]
        if progreso:
            progreso(t["nombre"], pagina, total_paginas, len(filas))
        if not datos or len(datos) < POR_PAGINA or (total_paginas and pagina >= total_paginas):
            break
        pagina += 1
    return a_dataframe(filas)


def buscar_en_todas(texto: str, max_paginas: int | None = 2) -> pd.DataFrame:
    """Busca el mismo material en todas las tiendas y ordena por precio (para comparar)."""
    frames = []
    for clave in TIENDAS:
        try:
            frames.append(productos(clave, buscar=texto, max_paginas=max_paginas))
        except requests.RequestException as e:
            print(f"{clave}: {e}")
    df = pd.concat(frames, ignore_index=True) if frames else a_dataframe([])
    return df.sort_values("precio", na_position="last").reset_index(drop=True)


def catalogo_completo(claves=None, max_paginas=None, progreso=None):
    """Devuelve (productos, categorias) de todas las tiendas (o de las indicadas en 'claves')."""
    prods, cats = [], []
    for clave in list(claves or TIENDAS):
        try:                                  # si una tienda falla, se avisa y se sigue con las demás
            cats.append(categorias(clave))
            prods.append(productos(clave, max_paginas=max_paginas, progreso=progreso))
        except requests.RequestException as e:
            print(f"AVISO: {TIENDAS[clave]['nombre']} no respondió ({type(e).__name__}); se omite.")
    if not prods:
        raise requests.ConnectionError("Ninguna tienda respondió.")
    return (pd.concat(prods, ignore_index=True) if prods else a_dataframe([]),
            pd.concat(cats, ignore_index=True) if cats else pd.DataFrame())


def mostrar_progreso(tienda, pagina, total, n):
    print(f"{tienda}: página {pagina}/{total} -> {n} productos", flush=True)

## 4. Exportación a Excel

`guardar_excel(productos, categorias, nombre)` crea `C:\APIS_PRECIOS\datos\<nombre>_AAAAMMDD_HHMMSS.xlsx` con:

| Hoja | Contenido |
|---|---|
| **Productos** | Una fila por producto: tienda, nombre, SKU, precio, precio regular, oferta, disponible, categorías, marca, observación, enlace |
| **Categorias** | Categorías de cada tienda con su cantidad de productos |
| **Resumen** | Por tienda y categoría: cantidad, precio mínimo, promedio y máximo |

Formato: encabezado fijo, filtros, ancho de columnas automático y precios con formato `$ #,##0`.

> Algunos productos tienen **precio 0**: la tienda los publica como "consultar precio". Se pueden filtrar en Excel.

In [ ]:
def guardar_excel(productos_df: pd.DataFrame, categorias_df: pd.DataFrame | None = None,
                  nombre: str = "precios_tiendas") -> Path:
    """Guarda los datos en un Excel con hojas Productos / Categorias / Resumen y devuelve la ruta."""
    nombre = re.sub(r"[^\w\-]+", "_", nombre).strip("_") or "precios"
    ruta = CARPETA_DATOS / f"{nombre}_{datetime.now():%Y%m%d_%H%M%S}.xlsx"
    with pd.ExcelWriter(ruta, engine="openpyxl") as xw:
        productos_df.to_excel(xw, sheet_name="Productos", index=False)
        if categorias_df is not None and not categorias_df.empty:
            categorias_df.to_excel(xw, sheet_name="Categorias", index=False)
        if not productos_df.empty and productos_df["precio"].notna().any():
            base = productos_df.assign(
                categoria=productos_df["categorias"].fillna("").str.split(" | ", regex=False).str[0])
            (base.groupby(["tienda", "categoria"])["precio"]
                 .agg(productos="count", precio_min="min", precio_promedio="mean", precio_max="max")
                 .reset_index()
                 .to_excel(xw, sheet_name="Resumen", index=False))
        for hoja in xw.sheets.values():                       # formato
            hoja.freeze_panes = "A2"
            hoja.auto_filter.ref = hoja.dimensions
            for col in hoja.columns:
                titulo = str(col[0].value or "")
                largo = max((len(str(c.value)) for c in col[:300] if c.value is not None), default=0)
                hoja.column_dimensions[col[0].column_letter].width = min(max(largo, len(titulo)) + 2, 60)
                if titulo.startswith("precio"):
                    for c in col[1:]:
                        c.number_format = '"$" #,##0'
    return ruta

## 5. Ensayo y descarga a Excel

### 5.1 Conexión con las tiendas
Debe mostrar `ok = True` y `estado_http = 200` en cada tienda.

In [ ]:
pd.DataFrame([probar_conexion(k) for k in TIENDAS])

### 5.2 Categorías de todas las tiendas

In [ ]:
def categorias_todas():
    """Categorías de todas las tiendas; si una no responde, avisa y continúa."""
    frames = []
    for k in TIENDAS:
        try:
            frames.append(categorias(k))
        except requests.RequestException as e:
            print(f"AVISO: {TIENDAS[k]['nombre']} no respondió ({type(e).__name__}); se omite.")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

df_categorias = categorias_todas()
print(len(df_categorias), "categorías")
df_categorias.head(20)

### 5.3 Buscar un material en todas las tiendas → Excel
Cambia el texto de `BUSQUEDA` para comparar otro material.

In [ ]:
BUSQUEDA = "cable 12"
df_busqueda = buscar_en_todas(BUSQUEDA)
print(len(df_busqueda), "resultados ->", guardar_excel(df_busqueda, nombre=f"busqueda_{BUSQUEDA}"))
df_busqueda[["tienda", "nombre", "sku", "precio", "disponible"]].head(15)

### 5.4 Catálogo completo → Excel
**Todos** los materiales y equipos con precio, de todas las tiendas y categorías.

In [ ]:
df_productos, df_cats = catalogo_completo(progreso=mostrar_progreso)
ruta_excel = guardar_excel(df_productos, df_cats, nombre="todos_materiales_precios")
print(f"\n{len(df_productos):,} productos guardados en: {ruta_excel}")
df_productos.groupby("tienda")["precio"].agg(productos="count", minimo="min", mediana="median", maximo="max")

## 6. API REST con Swagger

Levanta un servidor local en **http://127.0.0.1:8001** (corre en segundo plano mientras el notebook esté abierto).
Abrir **http://127.0.0.1:8001/docs** → elegir endpoint → *Try it out* → *Execute*.

### Endpoints que descargan Excel (los principales)

| Método | Ruta | Qué hace |
|---|---|---|
| GET | `/excel/todo` | **Todos los materiales y precios** de todas las tiendas → Excel (varios minutos). Opcional `tienda=nalelectricos` para una sola |
| GET | `/excel/buscar?q=breaker` | Un material buscado en todas las tiendas → Excel |
| GET | `/excel/categoria?tienda=edelco&categoria=industria` | Todos los productos de una categoría → Excel |

En Swagger aparece el botón **Download file**; además queda una copia en `C:\APIS_PRECIOS\datos\`.

### Endpoints de consulta (respuesta JSON)

| Método | Ruta | Qué hace |
|---|---|---|
| GET | `/tiendas` | Tiendas configuradas y estado de conexión |
| GET | `/tiendas/{tienda}/categorias` | Categorías de una tienda |
| GET | `/tiendas/{tienda}/productos?buscar=&categoria=&max_paginas=` | Productos y precios |
| GET | `/comparar?q=cable 12` | Mismo material en todas las tiendas, ordenado por precio |

Errores: `404` tienda o categoría inexistente · `502` la tienda no respondió.

In [ ]:
import uvicorn
from fastapi import FastAPI, HTTPException, Query
from fastapi.responses import FileResponse

app = FastAPI(
    title="API Precios Materiales Eléctricos",
    description="Precios de materiales y equipos eléctricos de tiendas colombianas (ver /tiendas). "
                "Los endpoints /excel/* devuelven el archivo Excel y guardan una copia en C:\\APIS_PRECIOS\\datos.",
    version="1.0.0",
)
_XLSX = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"


def _seguro(fn, *a, **k):
    # Traduce errores de las tiendas a respuestas HTTP claras.
    try:
        return fn(*a, **k)
    except KeyError as e:
        raise HTTPException(404, str(e).strip("'\""))
    except requests.RequestException as e:
        raise HTTPException(502, f"La tienda no respondió correctamente: {e}")


def _json(df):
    return json.loads(df.to_json(orient="records", force_ascii=False))


def _excel(ruta):
    return FileResponse(ruta, media_type=_XLSX, filename=ruta.name)


# ---------- Excel directo ----------
@app.get("/excel/todo", tags=["Excel"], summary="TODOS los materiales y precios -> Excel")
def excel_todo(tienda: str | None = Query(None, description="Vacío = todas. Claves: edelco, ineldec, nalelectricos, eimpsa, eys")):
    prods, cats = _seguro(catalogo_completo, [tienda] if tienda else None)
    return _excel(guardar_excel(prods, cats, nombre="todos_materiales_precios"))


@app.get("/excel/buscar", tags=["Excel"], summary="Buscar un material en todas las tiendas -> Excel")
def excel_buscar(q: str = Query(..., description="Ej. 'cable 12', 'breaker 20', 'transformador'"),
                 max_paginas: int = Query(5, ge=1, le=20)):
    return _excel(guardar_excel(_seguro(buscar_en_todas, q, max_paginas), nombre=f"busqueda_{q}"))


@app.get("/excel/categoria", tags=["Excel"], summary="Productos de una categoría -> Excel")
def excel_categoria(tienda: str, categoria: str = Query(..., description="Id o slug (ver /tiendas/{tienda}/categorias)")):
    df = _seguro(productos, tienda, None, categoria)
    return _excel(guardar_excel(df, nombre=f"{tienda}_{categoria}"))


# ---------- Consultas JSON ----------
@app.get("/tiendas", tags=["Consultas"], summary="Tiendas y estado de conexión")
def api_tiendas():
    return [{"clave": k, **probar_conexion(k)} for k in TIENDAS]


@app.get("/tiendas/{tienda}/categorias", tags=["Consultas"], summary="Categorías de una tienda")
def api_categorias(tienda: str):
    df = _seguro(categorias, tienda)
    return {"total": len(df), "categorias": _json(df)}


@app.get("/tiendas/{tienda}/productos", tags=["Consultas"], summary="Productos y precios de una tienda")
def api_productos(tienda: str, buscar: str | None = None, categoria: str | None = None,
                  max_paginas: int | None = Query(None, ge=1, description="100 productos por página; vacío = todas")):
    df = _seguro(productos, tienda, buscar, categoria, max_paginas)
    return {"total": len(df), "productos": _json(df)}


@app.get("/comparar", tags=["Consultas"], summary="Mismo material en todas las tiendas (ordenado por precio)")
def api_comparar(q: str, max_paginas: int = Query(2, ge=1, le=20)):
    df = _seguro(buscar_en_todas, q, max_paginas)
    return {"total": len(df), "productos": _json(df)}


# ---------- Arranque en segundo plano ----------
try:                                            # si se re-ejecuta la celda, detiene la API anterior
    servidor.should_exit = True
    time.sleep(1.5)
except NameError:
    pass
servidor = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PUERTO_API, log_level="warning"))
threading.Thread(target=servidor.run, daemon=True).start()
time.sleep(2)
print(f"API corriendo -> Swagger: http://127.0.0.1:{PUERTO_API}/docs")

## 7. Prueba automática de la API

Llama a la API como lo haría Swagger y verifica que devuelva un Excel válido.

In [ ]:
API = f"http://127.0.0.1:{PUERTO_API}"

def probar(ruta, params=None):
    t = time.time()
    r = requests.get(API + ruta, params=params, timeout=900)
    ok = r.ok and r.headers.get("content-type", "").startswith(_XLSX)
    filas = len(pd.read_excel(io.BytesIO(r.content), sheet_name="Productos")) if ok else 0
    print(f"[{'OK' if ok else 'FALLA'}] {ruta} {params or ''} -> HTTP {r.status_code}, {filas:,} productos, {time.time()-t:.0f}s")

print(requests.get(API + "/tiendas", timeout=60).json())
probar("/excel/buscar", {"q": "breaker"})
probar("/excel/categoria", {"tienda": "edelco", "categoria": "industria"})
probar("/excel/todo", {"tienda": "eimpsa"})   # una tienda, para no repetir la descarga completa de la sección 5.4

## 8. Mantenimiento

### Detener la API

In [ ]:
# Descomenta y ejecuta para detener la API:
# servidor.should_exit = True

### Agregar otra tienda
Sirven tiendas hechas en **WooCommerce** o **Shopify**. Para verificar, abre en el navegador:

- WooCommerce: `https://LA-TIENDA/wp-json/wc/store/v1/products?per_page=1`
- Shopify: `https://LA-TIENDA/products.json?limit=1`

Si muestra datos de un producto, agrégala en el diccionario `TIENDAS` de la sección 2:

```python
"nueva": {"nombre": "Nueva Tienda", "url": "https://nuevatienda.com", "tipo": "woocommerce", "precio_publicado": True},
```

> No se incluyó **Interelectricas**: su servidor bloquea las consultas automáticas (HTTP 403).

### Errores frecuentes

| Mensaje | Causa | Solución |
|---|---|---|
| `SSLError ... self signed certificate` | Proxy/antivirus revisa HTTPS | Ejecutar la sección 1 (instala `truststore`) y reiniciar el kernel |
| `ModuleNotFoundError` | Faltan librerías | Sección 1 + *Restart* + *Run All* |
| `address already in use` / puerto 8001 ocupado | La API ya está corriendo | Usar la que está abierta o cambiar `PUERTO_API` |
| `502` en Swagger / `ConnectionError`, `RemoteDisconnected` | La tienda cortó la conexión (saturada o muchas consultas seguidas) | El notebook reintenta solo 4 veces; si sigue, esperar unos minutos y volver a ejecutar. Las demás tiendas se descargan igual |
| Precio `0` | Producto publicado como "consultar precio" | Filtrar en Excel |